# Notebook 03 — Split the Data

## Objective

Prepare labeled data for machine learning by creating
training, validation, and test splits.

## Input

`artifacts/notebook_02/ml_table_labeled.parquet`

## Split

- Train: 70%
- Validation: 15%
- Test: 15%
- Stratified by `late`
- `random_state = 42`

## Important

Orders without a known `late` label are excluded from
supervised splits but remain in the Notebook 02 artifact.

## Grain

One row = one order

In [1]:
import pandas as pd
import numpy as np

from pathlib import Path

from sklearn.model_selection import train_test_split

In [2]:
input_path = Path("../artifacts/notebook_02/ml_table_labeled.parquet")

ml_table = pd.read_parquet(input_path)

print("Loaded labeled ML table successfully.")
print("Shape:", ml_table.shape)

Loaded labeled ML table successfully.
Shape: (99441, 22)


In [3]:
print("Rows:", len(ml_table))
print("Unique orders:", ml_table["order_id"].nunique())
print("Order IDs unique:", ml_table["order_id"].is_unique)

Rows: 99441
Unique orders: 99441
Order IDs unique: True


In [4]:
print("Label counts:")
print(
    ml_table["late"]
    .value_counts(dropna=False)
    .sort_index()
)

Label counts:
late
0.0    88649
1.0     7827
NaN     2965
Name: count, dtype: int64


In [5]:
labeled_data = ml_table[
    ml_table["late"].notna()
].copy()

unlabeled_data = ml_table[
    ml_table["late"].isna()
].copy()

print("Labeled orders:", len(labeled_data))
print("Unlabeled orders:", len(unlabeled_data))

Labeled orders: 96476
Unlabeled orders: 2965


In [6]:
assert labeled_data["late"].notna().all()

assert set(
    labeled_data["late"].unique()
).issubset({0.0, 1.0})

assert labeled_data["order_id"].is_unique

print("Labeled dataset validation passed.")

Labeled dataset validation passed.


In [7]:
y = labeled_data["late"].astype(int)

X = labeled_data.drop(columns=["late"])

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (96476, 21)
y shape: (96476,)


In [8]:
RANDOM_STATE = 42

TEST_SIZE = 0.15
VALIDATION_SIZE = 0.15

print("Random state:", RANDOM_STATE)
print("Validation size:", VALIDATION_SIZE)
print("Test size:", TEST_SIZE)

Random state: 42
Validation size: 0.15
Test size: 0.15


In [9]:
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

print("Train + Validation:", X_train_val.shape)
print("Test:", X_test.shape)

Train + Validation: (82004, 21)
Test: (14472, 21)


In [10]:
validation_ratio = VALIDATION_SIZE / (1 - TEST_SIZE)

X_train, X_validation, y_train, y_validation = train_test_split(
    X_train_val,
    y_train_val,
    test_size=validation_ratio,
    random_state=RANDOM_STATE,
    stratify=y_train_val,
)

print("Train:", X_train.shape)
print("Validation:", X_validation.shape)
print("Test:", X_test.shape)

Train: (67532, 21)
Validation: (14472, 21)
Test: (14472, 21)


In [11]:
train = X_train.copy()
train["late"] = y_train

validation = X_validation.copy()
validation["late"] = y_validation

test = X_test.copy()
test["late"] = y_test

print("Train shape:", train.shape)
print("Validation shape:", validation.shape)
print("Test shape:", test.shape)

Train shape: (67532, 22)
Validation shape: (14472, 22)
Test shape: (14472, 22)


In [12]:
train_orders = set(train["order_id"])
validation_orders = set(validation["order_id"])
test_orders = set(test["order_id"])

print("Train ∩ Validation:", len(train_orders & validation_orders))
print("Train ∩ Test:", len(train_orders & test_orders))
print("Validation ∩ Test:", len(validation_orders & test_orders))

Train ∩ Validation: 0
Train ∩ Test: 0
Validation ∩ Test: 0


In [13]:
assert train["order_id"].is_unique
assert validation["order_id"].is_unique
assert test["order_id"].is_unique

print("Grain validation passed for all splits.")

Grain validation passed for all splits.


In [14]:
def show_label_distribution(name, df):
    distribution = (
        df["late"]
        .value_counts(normalize=True)
        .sort_index()
        * 100
    )

    print(f"\n{name}")
    print(df["late"].value_counts().sort_index())
    print("Percentage:")
    print(distribution)


show_label_distribution("Train", train)
show_label_distribution("Validation", validation)
show_label_distribution("Test", test)


Train
late
0    62053
1     5479
Name: count, dtype: int64
Percentage:
late
0    91.886809
1     8.113191
Name: proportion, dtype: float64

Validation
late
0    13298
1     1174
Name: count, dtype: int64
Percentage:
late
0    91.887783
1     8.112217
Name: proportion, dtype: float64

Test
late
0    13298
1     1174
Name: count, dtype: int64
Percentage:
late
0    91.887783
1     8.112217
Name: proportion, dtype: float64


In [15]:
overall_late_ratio = y.mean()

train_late_ratio = train["late"].mean()
validation_late_ratio = validation["late"].mean()
test_late_ratio = test["late"].mean()

print("Overall late ratio:", overall_late_ratio)
print("Train late ratio:", train_late_ratio)
print("Validation late ratio:", validation_late_ratio)
print("Test late ratio:", test_late_ratio)

Overall late ratio: 0.08112898544715784
Train late ratio: 0.08113190783628502
Validation late ratio: 0.08112216694306247
Test late ratio: 0.08112216694306247


In [16]:
assert abs(train["late"].mean() - overall_late_ratio) < 0.001
assert abs(validation["late"].mean() - overall_late_ratio) < 0.001
assert abs(test["late"].mean() - overall_late_ratio) < 0.001

In [17]:
assert len(train) + len(validation) + len(test) == len(labeled_data)

print("Total split rows:",
      len(train) + len(validation) + len(test))

print("Labeled rows:",
      len(labeled_data))

print("Split accounting validation passed.")

Total split rows: 96476
Labeled rows: 96476
Split accounting validation passed.


In [18]:
split_orders = (
    set(train["order_id"])
    | set(validation["order_id"])
    | set(test["order_id"])
)

unlabeled_orders = set(unlabeled_data["order_id"])

print("Unlabeled orders:", len(unlabeled_orders))
print("Unlabeled orders in splits:",
      len(unlabeled_orders & split_orders))

Unlabeled orders: 2965
Unlabeled orders in splits: 0


In [19]:
assert len(unlabeled_orders & split_orders) == 0

print("Unlabeled orders exclusion validation passed.")

Unlabeled orders exclusion validation passed.


## Leakage Prevention

The following columns are not predictive features:

- `late` — target
- `delivery_delay_days` — target-derived
- `label_data_available` — label availability
- `order_delivered_customer_date` — outcome timestamp

`order_id` is retained only as an identifier.

## Date Range by Split

Because the data is randomly and stratified-split,
date ranges may overlap across Train, Validation, and Test.

This check is descriptive only.

In [20]:
date_column = "order_purchase_timestamp"

print("Date range by split:\n")

for name, X_split in [
    ("Train", X_train),
    ("Validation", X_validation),
    ("Test", X_test),
]:
    dates = pd.to_datetime(X_split[date_column], errors="coerce")

    print(f"{name}:")
    print("  Min date:", dates.min())
    print("  Max date:", dates.max())
    print("  Missing dates:", dates.isna().sum())
    print()

Date range by split:

Train:
  Min date: 2016-10-03 09:44:50
  Max date: 2018-08-29 15:00:37
  Missing dates: 0

Validation:
  Min date: 2016-09-15 12:16:38
  Max date: 2018-08-29 10:22:35
  Missing dates: 0

Test:
  Min date: 2016-10-03 21:13:36
  Max date: 2018-08-29 14:18:23
  Missing dates: 0



In [21]:
date_range_summary = pd.DataFrame({
    "split": ["Train", "Validation", "Test"],
    "min_purchase_date": [
        train[date_column].min(),
        validation[date_column].min(),
        test[date_column].min(),
    ],
    "max_purchase_date": [
        train[date_column].max(),
        validation[date_column].max(),
        test[date_column].max(),
    ],
})

date_range_summary

,split,min_purchase_date,max_purchase_date
0,Train,2016-10-03 09:44:50,2018-08-29 15:00:37
1,Validation,2016-09-15 12:16:38,2018-08-29 10:22:35
2,Test,2016-10-03 21:13:36,2018-08-29 14:18:23


In [22]:
assert date_range_summary["min_purchase_date"].notna().all()
assert date_range_summary["max_purchase_date"].notna().all()

assert (
    date_range_summary["min_purchase_date"]
    <= date_range_summary["max_purchase_date"]
).all()

print("Date range validation passed.")

Date range validation passed.


In [23]:
artifact_dir = Path("../artifacts/notebook_03")
artifact_dir.mkdir(parents=True, exist_ok=True)

print("Artifact directory:", artifact_dir)

Artifact directory: ..\artifacts\notebook_03


In [24]:
train_path = artifact_dir / "train.parquet"
validation_path = artifact_dir / "validation.parquet"
test_path = artifact_dir / "test.parquet"

train.to_parquet(train_path, index=False)
validation.to_parquet(validation_path, index=False)
test.to_parquet(test_path, index=False)

print("Saved:")
print(train_path)
print(validation_path)
print(test_path)

Saved:
..\artifacts\notebook_03\train.parquet
..\artifacts\notebook_03\validation.parquet
..\artifacts\notebook_03\test.parquet


In [25]:
saved_train = pd.read_parquet(train_path)
saved_validation = pd.read_parquet(validation_path)
saved_test = pd.read_parquet(test_path)

print("Train:", saved_train.shape)
print("Validation:", saved_validation.shape)
print("Test:", saved_test.shape)

Train: (67532, 22)
Validation: (14472, 22)
Test: (14472, 22)


In [26]:
assert saved_train["order_id"].is_unique
assert saved_validation["order_id"].is_unique
assert saved_test["order_id"].is_unique

assert len(saved_train) + len(saved_validation) + len(saved_test) == 96476

assert (
    set(saved_train["order_id"])
    & set(saved_validation["order_id"])
    == set()
)

assert (
    set(saved_train["order_id"])
    & set(saved_test["order_id"])
    == set()
)

assert (
    set(saved_validation["order_id"])
    & set(saved_test["order_id"])
    == set()
)

print("Final split validation passed.")

Final split validation passed.


## Summary

- Loaded the labeled ML table.
- Separated labeled and unlabeled orders.
- Created stratified Train/Validation/Test splits.
- Preserved the one-row-per-order grain.
- Verified no overlap between splits.
- Saved the three Parquet datasets.

### Split

- Train: 67,532
- Validation: 14,472
- Test: 14,472

### Output

- `artifacts/notebook_03/train.parquet`
- `artifacts/notebook_03/validation.parquet`
- `artifacts/notebook_03/test.parquet`